# Dr. Agent Model Tutorial with MIMIC-IV

This notebook demonstrates how to use the Dr. Agent model for mortality prediction on MIMIC-IV data. Dr. Agent uses two reinforcement learning agents with dynamic skip connections to capture long-term dependencies in patient EHR sequences.

**Paper:** Gao et al. "Dr. Agent: Clinical predictive model via mimicked second opinions" (JAMIA 2020)


# 1. Environment Setup

Configure deterministic behaviour and import the libraries required for the tutorial.

In [1]:
import random
import numpy as np
import torch

from pyhealth.datasets import MIMIC4Dataset
from pyhealth.datasets.splitter import split_by_patient
from pyhealth.datasets.utils import get_dataloader
from pyhealth.tasks.mortality_prediction import MortalityPredictionMIMIC4

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cpu


# 2. Load MIMIC-IV Dataset
Point to the preprocessed MIMIC-IV tables and load the dataset.

In [2]:
dataset = MIMIC4Dataset(
    ehr_root="C:\\Users\\Eli\\Data\\physionet.org\\files\\mimiciv\\3.1\\",  # Update this path
    ehr_tables=[
        "patients",
        "admissions",
        "diagnoses_icd",
        "procedures_icd",
        "prescriptions",
    ],
    dev=True,  # Set to False for full dataset
)

Memory usage Starting MIMIC4Dataset init: 465.3 MB
Initializing mimic4 dataset from C:\Users\Eli\Data\physionet.org\files\mimiciv\3.1\|None|None (dev mode: True)
Initializing MIMIC4EHRDataset with tables: ['patients', 'admissions', 'diagnoses_icd', 'procedures_icd', 'prescriptions'] (dev mode: True)
No cache_dir provided. Using default cache dir: C:\Users\Eli\AppData\Local\pyhealth\pyhealth\Cache\d3e587c7-9e96-51f1-a45b-838106f53f68
Using default EHR config: C:\Python\PyHealth\pyhealth\datasets\configs\mimic4_ehr.yaml
Memory usage Before initializing mimic4_ehr: 466.4 MB
Duplicate table names in tables list. Removing duplicates.
Initializing mimic4_ehr dataset from C:\Users\Eli\Data\physionet.org\files\mimiciv\3.1\ (dev mode: True)
Memory usage After initializing mimic4_ehr: 466.4 MB
Memory usage After EHR dataset initialization: 466.4 MB
Memory usage Completed MIMIC4Dataset init: 466.4 MB


# 3. Prepare PyHealth Dataset
Use the built-in `MortalityPredictionMIMIC4` task to convert patients into labeled visit samples and split them into training, validation, and test subsets.

In [4]:
task = MortalityPredictionMIMIC4()
sample_dataset = dataset.set_task(task)

print(f"Total task samples: {len(sample_dataset)}")
print(f"Input schema: {sample_dataset.input_schema}")
print(f"Output schema: {sample_dataset.output_schema}")

if len(sample_dataset) == 0:
    raise RuntimeError(
        "The task did not produce any samples. "
        "Disable dev mode or adjust table selections."
    )

train_ds, val_ds, test_ds = split_by_patient(
    sample_dataset, [0.7, 0.1, 0.2], seed=SEED
)
print(f"Train/Val/Test sizes: {len(train_ds)}, {len(val_ds)}, {len(test_ds)}")

Setting task MortalityPredictionMIMIC4 for mimic4 base dataset...
Fitting processors on the dataset...
Label mortality vocab: {0: 0, 1: 1}
Processing samples and saving to C:\Users\Eli\AppData\Local\pyhealth\pyhealth\Cache\d3e587c7-9e96-51f1-a45b-838106f53f68\tasks\MortalityPredictionMIMIC4\samples_19be41a9-b01f-48bb-af79-94b5f6083c95.ld...
Applying processors on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 860 samples. (0 to 860)


  0%|          | 0/860 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 860/860 [00:00<00:00, 8644.55it/s]

Worker 0 finished processing samples.
Cached processed samples to C:\Users\Eli\AppData\Local\pyhealth\pyhealth\Cache\d3e587c7-9e96-51f1-a45b-838106f53f68\tasks\MortalityPredictionMIMIC4\samples_19be41a9-b01f-48bb-af79-94b5f6083c95.ld
Total task samples: 860
Input schema: {'conditions': 'sequence', 'procedures': 'sequence', 'drugs': 'sequence'}
Output schema: {'mortality': 'binary'}
Train/Val/Test sizes: 615, 63, 182


# 4. Inspect Batch Structure

Build PyHealth dataloaders and verify the keys and tensor shapes before training.

In [5]:
BATCH_SIZE = 32

train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=BATCH_SIZE) if len(val_ds) else None
test_loader = get_dataloader(test_ds, batch_size=BATCH_SIZE) if len(test_ds) else None

if len(train_loader) == 0:
    raise RuntimeError(
        "The training loader is empty. "
        "Increase the dataset size or adjust the task configuration."
    )

first_batch = next(iter(train_loader))


def describe(value):
    if hasattr(value, "shape"):
        return f"{type(value).__name__}(shape={tuple(value.shape)})"
    if isinstance(value, (list, tuple)):
        return f"{type(value).__name__}(len={len(value)})"
    return type(value).__name__


batch_summary = {key: describe(value) for key, value in first_batch.items()}
print("Batch structure:")
for key, desc in batch_summary.items():
    print(f"  {key}: {desc}")

Batch structure:
  visit_id: list(len=32)
  patient_id: list(len=32)
  conditions: Tensor(shape=(32, 25))
  procedures: Tensor(shape=(32, 8))
  drugs: Tensor(shape=(32, 99))
  mortality: Tensor(shape=(32, 1))


# 5. Instantiate Dr. Agent Model

Create the PyHealth Agent model with custom hyperparameters. The model uses:
- **Primary agent**: Observes current visit to learn current health status
- **Second-opinion agent**: Considers entire patient history for a global view
- **Dynamic skip connections**: Selects optimal historical states via policy gradient

In [6]:
from pyhealth.models import Agent

model = Agent(
    dataset=sample_dataset,
    embedding_dim=128,
    hidden_dim=128,
    # Agent-specific hyperparameters
    n_actions=10,      # History window size (K in paper)
    n_units=64,        # Agent MLP hidden units
    dropout=0.5,       # Dropout rate
    lamda=0.5,         # Skip connection weight
    cell="gru",        # RNN cell type: "gru" or "lstm"
    use_baseline=True, # Use baseline for variance reduction
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Feature keys: {model.feature_keys}")
print(f"Label key: {model.label_key}")
print(f"Mode: {model.mode}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

C:\Python\PyHealth\pyhealth\sampler\sage_sampler.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature keys: ['conditions', 'procedures', 'drugs']
Label key: mortality
Mode: binary
Total parameters: 914,499
Trainable parameters: 914,499


# 6. Verify Forward Pass

Run a single forward pass to ensure the model works correctly before training.

In [7]:
model.eval()
with torch.no_grad():
    sample_batch = {k: v.to(device) if hasattr(v, "to") else v for k, v in first_batch.items()}
    output = model(**sample_batch)

print("Forward pass output keys:", list(output.keys()))
print(f"Loss: {output['loss'].item():.4f}")
print(f"y_prob shape: {output['y_prob'].shape}")
print(f"y_true shape: {output['y_true'].shape}")

Forward pass output keys: ['loss', 'y_prob', 'y_true', 'logit']
Loss: 3.8830
y_prob shape: torch.Size([32, 1])
y_true shape: torch.Size([32, 1])


# 7. Configure Trainer

Wrap the model with the PyHealth `Trainer` to handle optimisation, gradient clipping, and metric logging.

In [8]:
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["roc_auc", "pr_auc"],
    device=str(device),
    enable_logging=True,
)

training_config = {
    "epochs": 20,
    "optimizer_params": {"lr": 1e-3},
    "max_grad_norm": 5.0,
    "monitor": "roc_auc",
}

Agent(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(2597, 128, padding_idx=0)
    (procedures): Embedding(870, 128, padding_idx=0)
    (drugs): Embedding(932, 128, padding_idx=0)
  ))
  (agent): ModuleDict(
    (conditions): AgentLayer(
      (agent1_fc1): Linear(in_features=128, out_features=64, bias=True)
      (agent1_fc2): Linear(in_features=64, out_features=10, bias=True)
      (agent2_fc1): Linear(in_features=128, out_features=64, bias=True)
      (agent2_fc2): Linear(in_features=64, out_features=10, bias=True)
      (agent1_value): Linear(in_features=64, out_features=1, bias=True)
      (agent2_value): Linear(in_features=64, out_features=1, bias=True)
      (rnn): GRUCell(128, 128)
      (nn_dropout): Dropout(p=0.5, inplace=False)
      (softmax): Softmax(dim=1)
      (tanh): Tanh()
    )
    (procedures): AgentLayer(
      (agent1_fc1): Linear(in_features=128, out_features=64, bias=True)
      (agent1_fc2): Linear(in_features=64, 

# 8. Train the Model

Run multiple epochs with gradient clipping, scheduler updates, and logging of loss/metrics per epoch.

In [9]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    **training_config,
)

Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.001}
Weight decay: 0.0
Max grad norm: 5.0
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x0000022A31572E90>
Monitor: roc_auc
Monitor criterion: max
Epochs: 20
Patience: None



Epoch 0 / 20: 100%|██████████| 20/20 [00:19<00:00,  1.04it/s]

--- Train epoch-0, step-20 ---
loss: 3.1900



Evaluation: 100%|██████████| 2/2 [00:01<00:00,  1.69it/s]

--- Eval epoch-0, step-20 ---
roc_auc: nan
pr_auc: 0.0000
loss: 3.7248



...


C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
Epoch 19 / 20: 100%|██████████| 20/20 [00:22<00:00,  1.11s/it]

--- Train epoch-19, step-400 ---
loss: -0.3181



Evaluation: 100%|██████████| 2/2 [00:01<00:00,  1.99it/s]

--- Eval epoch-19, step-400 ---
roc_auc: nan
pr_auc: 0.0000
loss: -1.4392



C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


# 9. Evaluate on Test Split

Switch to evaluation mode, collect predictions for validation and test splits, and compute metrics.

In [10]:
evaluation_results = {}

for split_name, loader in [("validation", val_loader), ("test", test_loader)]:
    if loader is None:
        print(f"Skipping {split_name} (no data)")
        continue
    
    metrics = trainer.evaluate(loader)
    evaluation_results[split_name] = metrics
    
    formatted = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items())
    print(f"{split_name.title()} metrics: {formatted}")

Evaluation: 100%|██████████| 2/2 [00:00<00:00,  2.94it/s]
C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\sklearn\metrics\_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
C:\Users\Eli\miniconda3\envs\MyTestProj\Lib\site-packages\sklearn\metrics\_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Validation metrics: roc_auc=nan, pr_auc=0.0000, loss=-1.3267


Evaluation: 100%|██████████| 6/6 [00:01<00:00,  4.50it/s]

Test metrics: roc_auc=0.2576, pr_auc=0.0211, loss=-0.8185


# 10. Inspect Sample Predictions

Run inference to preview predicted probabilities alongside ground-truth labels.

In [11]:
target_loader = test_loader if test_loader is not None else val_loader
if target_loader is None:
    target_loader = train_loader

y_true, y_prob, mean_loss = trainer.inference(target_loader)

# For binary classification, get positive class probability
positive_prob = y_prob if y_prob.ndim == 1 else y_prob[..., -1]

print(f"Mean inference loss: {mean_loss:.4f}")
print(f"\nSample predictions (first 10):")
print("-" * 40)
print(f"{'True Label':<12} {'Pred Prob':<12}")
print("-" * 40)

for label, prob in zip(y_true[:10].tolist(), positive_prob[:10].tolist()):
    print(f"{int(label):<12} {prob:.4f}")

Evaluation: 100%|██████████| 6/6 [00:01<00:00,  3.43it/s]


Mean inference loss: -0.8386

Sample predictions (first 10):
----------------------------------------
True Label   Pred Prob   
----------------------------------------


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'

# 11. Save Model Checkpoint

In [1]:
checkpoint_path = "agent_mimic4_mortality.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "training_config": training_config,
    "evaluation_results": evaluation_results,
}, checkpoint_path)

print(f"Model saved to: {checkpoint_path}")

NameError: name 'torch' is not defined